# Modern Application Development – I: Comprehensive Lecture Notes  
**Professor Nitin Chandrachoodan & Professor Thejesh G N, IIT Madras**  
**Week 6: API Design, REST, OpenAPI Specification & Flask-RESTful**

---

## Table of Contents
1. [Introduction to API Design and REST](#1-introduction-to-api-design-and-rest)
2. [The Constraints of REST](#2-the-constraints-of-rest)
3. [REST in Practice: HTTP Verbs, Idempotency, and State Transfer](#3-rest-in-practice)
4. [REST API Examples – Wikipedia](#4-rest-api-examples--wikipedia)
5. [REST API Examples – CoWin](#5-rest-api-examples--cowin)
6. [The OpenAPI Specification (OAS)](#6-the-openapi-specification-oas)
7. [Important Concepts of an API (OAS Structure in Depth)](#7-important-concepts-of-an-api)
8. [Documenting an API with Swagger Editor (Practical)](#8-documenting-an-api-with-swagger-editor)
9. [Building REST APIs with Flask-RESTful](#9-building-rest-apis-with-flask-restful)

---

## 1. Introduction to API Design and REST

### 1.1 The Role of APIs in Web Applications
An **Application Programming Interface (API)** defines a set of rules, protocols, and tools that allow different software components to communicate. In the context of web applications, a **Web API** is an interface exposed by a server that allows clients (browsers, mobile apps, other servers) to request data and perform actions over HTTP. The API sits between the Model (data) and the Controller/View layers, enabling a clean separation of concerns.

APIs achieve **information hiding**: the client does not need to know whether the server uses MySQL, PostgreSQL, or a flat‑file system. It only knows the API’s endpoints, the data formats (usually JSON), and the expected behaviour. This allows the server implementation to evolve independently without breaking clients, as long as the API contract is honoured.

### 1.2 Distributed Software Architecture and the Web
A web application is inherently a **distributed system** because the client and server reside on different machines, often thousands of kilometres apart. Designing such a system involves making assumptions (or avoiding them) about network reliability, latency, authentication, and state. The web was originally a simple document‑retrieval system (HTTP GET for static HTML), but it has grown into a platform for complex, interactive applications. This evolution demanded a consistent architectural style to guide the design of scalable, maintainable web services.

### 1.3 Roy Fielding and the Birth of REST
**REST** stands for **REpresentational State Transfer**. It was introduced by **Roy Fielding** in his doctoral dissertation (2000) at the University of California, Irvine. Fielding was a principal author of the HTTP specification and a co‑founder of the Apache HTTP Server project. His thesis analysed the architectural properties that made the World Wide Web successful and formalised them into an **architectural style** – a set of constraints that, when applied, yield desirable non‑functional properties such as scalability, modifiability, and performance.

**REST is not a protocol**, nor is it a standard. It is a style, much like the MVC pattern is a design idea. You do not “implement REST” by installing a library; you design your application’s architecture to satisfy the REST constraints. The most common way to realise RESTful systems today is over HTTP, but REST itself is transport‑agnostic.

---

## 2. The Constraints of REST

Fielding’s thesis defines six constraints that characterise a RESTful architecture. Each constraint is motivated by a particular engineering concern.

### 2.1 Client‑Server
The system is composed of clients and servers. The server stores data and performs computation; the client initiates requests and presents results to the user. This separation of concerns allows the user interface to evolve independently of the data storage logic, and it enables multiple different clients (web, mobile, desktop) to share the same server.

### 2.2 Statelessness
**No client context is stored on the server between requests.** Each request from the client to the server must contain all the information necessary to understand and process the request. The server cannot rely on any stored conversation state. Session state, if needed, is kept entirely on the client (e.g., in cookies, tokens, or URL parameters) and sent with each request.

**Why this is critical on the web:**
- The server may be restarted, replaced, or load‑balanced across multiple instances without losing any session.
- A request can be routed to any available server in a cluster because no server holds a unique session for that client.
- It simplifies the server design and improves scalability.

**Implications:** The client must resend authentication credentials (like an API key or JWT token) with every request, and any state that spans multiple requests must be managed by the client and transmitted explicitly.

### 2.3 Cacheability
Responses must implicitly or explicitly label themselves as cacheable or non‑cacheable. If a response is cacheable, the client or an intermediate proxy (e.g., a CDN, a reverse proxy) is allowed to reuse that response data for later equivalent requests, potentially eliminating the need to contact the origin server.

**Why this matters:** Caching can dramatically reduce latency and server load. HTTP provides standardised headers (`Cache-Control`, `Expires`, `ETag`) that enable fine‑grained control over caching behaviour. A well‑designed RESTful API leverages these headers to let intermediaries cache responses when appropriate, without breaking the application’s semantics.

### 2.4 Layered System
A client cannot ordinarily tell whether it is connected directly to the end server or to an intermediary. The architecture can be composed of hierarchical layers: load balancers, authentication gateways, caching proxies, or middleware that enrich or transform requests. Each layer only interacts with its immediate neighbours.

**Benefits:**
- Security: a gateway layer can handle authentication, SSL termination, and rate limiting.
- Performance: caching layers can serve responses without touching the application servers.
- Evolvability: new layers (e.g., a logging layer, a request enrichment layer) can be added transparently without affecting the client.

This layered constraint is visible in HTTP’s ability to insert `Via` headers and the fact that proxies can be chained without the client’s awareness.

### 2.5 Uniform Interface
This is the most distinguishing feature of REST. It simplifies and decouples the architecture by providing a **consistent, uniform** way for clients to interact with resources across the entire system. The uniform interface is further decomposed into four sub‑constraints:

1. **Identification of resources:** Each resource (a concept, e.g., a user, an article) is uniquely identified by a URI (Uniform Resource Identifier). The URI is the name; the resource is the thing. Any piece of information that can be named can be a resource.

2. **Manipulation of resources through representations:** A client never interacts with the resource directly (it does not access a database row). Instead, the server sends a **representation** of the resource’s state (e.g., a JSON document, an HTML page). When a client wants to modify the resource, it sends a new representation back to the server. The same resource can have multiple representations (JSON, XML, HTML) based on content negotiation.

3. **Self‑descriptive messages:** Each message (request or response) contains all the information needed to process it. For example, the `Content-Type` header tells the recipient how to interpret the body; HTTP methods indicate the desired action. This eliminates the need for out‑of‑band documentation for basic processing.

4. **Hypermedia as the Engine of Application State (HATEOAS):** In a fully RESTful system, a client interacts with the application entirely through hypermedia provided dynamically by the server. The client starts at a well‑known entry point (e.g., the root URL). The response includes links to other available resources and actions. The client follows these links to transition to new states. The server can change the URI structure or add new resources without breaking clients, because clients are not hard‑coded to specific URIs. (This is the least commonly implemented constraint in practice, but it is the most powerful in terms of decoupling.)

### 2.6 Code on Demand (Optional)
Servers can temporarily extend client functionality by transferring executable code (e.g., JavaScript, Java applets). This allows the client to perform some computation locally, reducing the number of round‑trips. In modern web apps, this is ubiquitous (JavaScript running in the browser), but it is the only optional constraint in REST.

**Summary of REST’s value:** By adhering to these constraints, a system gains properties like scalability (stateless, cacheable), visibility (self‑descriptive messages), reliability, and modifiability. Understanding these constraints is fundamental to designing web APIs that are clean, scalable, and easy to evolve.

---

## 3. REST in Practice: HTTP Verbs, Idempotency, and State Transfer

### 3.1 How REST Maps to HTTP
Although REST is not tied to HTTP, HTTP provides a natural implementation. The following mapping is conventional:

- **Resources** are identified by URLs (a subset of URIs).
- **Representations** are documents in formats like JSON, XML, or HTML, indicated by `Content-Type`.
- **Operations** are expressed using HTTP methods (verbs):
  - `GET` – retrieve a representation of a resource.
  - `POST` – submit data to be processed; often creates a new resource.
  - `PUT` – replace a resource entirely, or create it if it doesn’t exist (client specifies the URI).
  - `PATCH` – partially modify a resource.
  - `DELETE` – remove a resource.

### 3.2 Idempotent vs. Non‑Idempotent Methods
An operation is **idempotent** if performing it multiple times has the same side‑effect as performing it once. This is a crucial concept for designing robust APIs.

- **Safe and idempotent:** `GET`, `HEAD`, `OPTIONS` – they do not modify server state.
- **Not safe but idempotent:** `PUT`, `DELETE`. If you `PUT` the same resource twice with the same body, the state remains the same after both requests. If you `DELETE` a resource twice, the second request still results in the resource being absent, without additional side effects.
- **Not idempotent:** `POST`. Sending the same `POST` request twice may create two resources (e.g., two identical orders), or trigger two payments.

Understanding idempotency helps in designing fault‑tolerant clients: if a network failure occurs, a `PUT` or `DELETE` can be safely retried, but a `POST` should be retried only if the client can verify that the previous request did not succeed.

### 3.3 The Meaning of “Representational State Transfer”
The term itself describes the mechanism: the client interacts with the application by exchanging **representations** of resource **state**. A client requests the current state of a resource (GET); the server returns a representation. To change the state, the client sends a representation to the server (POST/PUT). The state is thus *transferred* via representations. Because the server is stateless, each request carries the complete context needed for that state transition. The “state” is the application state (e.g., “I am on page 2 of the search results, looking at item 42”), and it is managed by the client, driven by hypermedia links.

### 3.4 CRUD vs. REST – A Common Misconception
**CRUD** (Create, Read, Update, Delete) is a database operation pattern. REST is an architectural style for networked applications. CRUD operations map nicely onto HTTP methods (`POST` → Create, `GET` → Read, `PUT`/`PATCH` → Update, `DELETE` → Delete), but they are not the same. A RESTful API can expose actions that are not pure CRUD (e.g., “approve”, “ship”, “reboot”), often expressed as POST requests to action‑named resources. CRUD is a convenient subset of what REST can express, but it is not the entirety of REST.

---

## 4. REST API Examples – Wikipedia

### 4.1 Why Wikipedia Provides a REST API
Wikipedia, as a massive collaborative knowledge base, needs to serve not only human readers but also automated tools, bots, researchers, and third‑party apps. A REST API allows these consumers to query, retrieve, and analyse Wikipedia’s content programmatically without scraping HTML pages. It separates the data (model) from the presentation (view), returning structured JSON that can be processed by any client.

### 4.2 Anatomy of a Wikipedia REST API Request
The lecture demonstrates searching for the article “Earth” using `curl`:
```
curl https://en.wikipedia.org/w/rest.php/v1/search/page?q=Earth&limit=1
```
**Breakdown:**
- `en.wikipedia.org` – the host (English Wikipedia).
- `/w/rest.php` – the entry point for the REST API (distinct from the normal web interface).
- `/v1` – API version. Versioning prevents breaking existing clients when the API evolves.
- `/search/page` – the specific resource/endpoint: a search over pages.
- `?q=Earth&limit=1` – query string parameters. `q` is the search query (required), `limit` is optional (default probably higher).

The response is a JSON object containing an array of `pages`. Each page object has an `id`, `key` (URL‑friendly title), `title` (human‑readable), `excerpt` (HTML snippet), `description` (metadata), and a `thumbnail` URL. This structured data is far easier for a program to consume than parsing the full HTML page.

### 4.3 Wikipedia API Documentation
The Wikipedia REST API documentation (accessible at `en.wikipedia.org/api/rest_v1/`) provides everything a developer needs:
- **Endpoint (route):** e.g., `/search/page`.
- **Method:** `GET`.
- **Parameters:** `q` (required string), `limit` (optional integer, default and max values specified).
- **Response schema:** describes exactly what fields are returned, their types, and when they might be `null`.
- **Status codes:** `200` (success, array of results), `200` with empty array if nothing found, `400` if `q` is missing, `500` for server errors.

This documentation style — listing the route, method, parameters, possible responses with schemas — is the essence of a well‑documented API and is the precursor to the machine‑readable OpenAPI Specification.

---

## 5. REST API Examples – CoWin

### 5.1 The CoWin API
CoWin (COVID Vaccine Intelligence Network) is the Indian government’s platform for vaccine registration and appointment scheduling. It exposes a public API to allow third‑party apps (like Paytm, HealthifyMe, etc.) to query availability and, with authentication, book slots. The API is a real‑world example of a high‑traffic, security‑sensitive REST API.

### 5.2 Key Endpoints
- **Unauthenticated endpoints:** e.g., `GET /v2/appointment/sessions/public/findByPin` – requires `pincode` and `date` parameters. Returns a JSON array of sessions at vaccination centres, each with centre details, available capacity, vaccine type, and age limits.
- **Authenticated endpoints:** e.g., downloading a vaccination certificate requires a bearer token obtained via OTP authentication. The API uses `BearerAuth` in the HTTP `Authorization` header.

### 5.3 Important Lessons from CoWin
- **Always respect rate limits and the API’s purpose.** CoWin explicitly warns against automated polling that could overload the servers. As a developer, you must design clients that are considerate (add delays, cache responses, avoid infinite loops). This is an ethical and practical requirement for any public API.
- **Authentication is layered.** Public data can be fetched without a token; private operations (booking, certificate download) require a token that proves the user’s identity. This token is sent with every request (statelessness).
- **Versioning** is visible in the URL path (`/v2/`), allowing the API to evolve without breaking existing applications.

---

## 6. The OpenAPI Specification (OAS)

### 6.1 The Need for a Standard API Description Language
Documentation written in prose is ambiguous, can become outdated, and requires human interpretation. A **machine‑readable API description** solves these problems. If the API is described in a standard, structured format, then:
- Tools can generate **interactive documentation** (like Swagger UI).
- Client libraries and server stubs can be **auto‑generated**.
- **Validation** of requests and responses against the spec can be automated.
- The specification can be version‑controlled alongside the code, serving as a **single source of truth**.

### 6.2 History: Swagger → OpenAPI
**Swagger** was an open‑source framework created by SmartBear Software. The **Swagger 2.0** specification became the basis for the **OpenAPI Specification (OAS)**, which is now governed by the OpenAPI Initiative under the Linux Foundation. **OAS 3.0** and later **3.1** are the current versions. The terms “OpenAPI” and “Swagger” are often used interchangeably, though “Swagger” more properly refers to the tooling (Swagger Editor, Swagger UI, Swagger Codegen).

### 6.3 What OAS Describes
OAS is **HTTP‑centric** and describes RESTful APIs. A compliant document defines:
- **General info:** title, version, description, contact, license.
- **Servers:** one or more base URLs (e.g., production, staging, localhost).
- **Paths:** the available endpoints, each with supported HTTP methods.
- **Operations:** for each method, a summary, description, parameters (in path, query, header, cookie), request body (with schema), and possible responses (status codes, headers, content type, schema).
- **Components:** reusable schemas (data models), parameters, responses, security schemes.

### 6.4 The Design‑First Philosophy
A core motivation for OAS is to promote **design‑first** development: write the API specification before writing any code. This allows stakeholders to review and agree on the contract early, and developers on both server and client sides can work in parallel. It reduces integration surprises because both sides are building to the same unambiguous spec. The alternative (code‑first, then document) often leads to inconsistent, outdated documentation.

---

## 7. Important Concepts of an API (OAS Structure in Depth)

### 7.1 YAML as a Specification Language
OAS documents can be written in **JSON** or **YAML**. YAML is more human‑friendly because it uses indentation instead of braces and allows comments. A minimal YAML OAS document:
```yaml
openapi: 3.1.0
info:
  title: Minimal OpenAPI Document
  version: 0.0.1
paths: {}
```
- `openapi` specifies the OAS version.
- `info` contains metadata.
- `paths` is where endpoints are defined (empty for a stub).

### 7.2 Defining Paths, Methods, and Operations
Each **path** (relative URL) is a key under `paths`. Under that path, you list the HTTP methods (`get`, `post`, etc.) as keys. Each method is an **operation** object containing:
- `summary` – short description.
- `description` – longer, can include Markdown.
- `parameters` – list of expected inputs (path, query, header, cookie), each with `name`, `in` (location), `required`, `schema` (type).
- `requestBody` – for POST/PUT/PATCH, the expected body structure.
- `responses` – a map of HTTP status codes to response objects.

**Example:**
```yaml
paths:
  /board:
    get:
      summary: Retrieve the current state of the board
      parameters: []
      responses:
        '200':
          description: Successful request
          content:
            application/json:
              schema:
                $ref: '#/components/schemas/Board'
        '404':
          description: Board not found
```

### 7.3 Parameters in Detail
Parameters can appear in:
- **path** (e.g., `/users/{id}`) – required, part of the URL.
- **query** (e.g., `?q=search`) – often optional, filters.
- **header** – custom HTTP headers.
- **cookie** – cookie values.

Each parameter has a `schema` specifying its data type (`string`, `integer`, etc.) and optionally `enum`, `default`, `format`. The `required` flag is mandatory for path parameters.

### 7.4 Request Body and Content Types
When a method expects a body (e.g., `POST` to create a user), the `requestBody` object specifies the `content` types supported (e.g., `application/json`). Each content type has a `schema` that describes the expected JSON structure, often referencing a component schema.

### 7.5 Responses and Status Codes
Under `responses`, you can specify multiple status codes. Each response can have:
- `description` – human‑readable.
- `content` – the response body’s media type and schema.
- `headers` – custom response headers.

Common codes:
- `200 OK` – successful retrieval.
- `201 Created` – successful resource creation (often with a `Location` header).
- `204 No Content` – successful deletion.
- `400 Bad Request` – client error (invalid input).
- `401 Unauthorized` – missing authentication.
- `404 Not Found` – resource does not exist.
- `500 Internal Server Error` – server failure.

### 7.6 Schemas and Reusable Components
The `components` section allows defining reusable schemas, parameters, and responses. A schema describes a data model using JSON Schema (e.g., an object with properties, types, required fields). Example:
```yaml
components:
  schemas:
    User:
      type: object
      properties:
        user_id:
          type: integer
        username:
          type: string
        email:
          type: string
      required:
        - username
        - email
```
Operations can then reference this schema with `$ref: '#/components/schemas/User'`.

### 7.7 The Power of a Machine‑Readable Spec
Because OAS is fully structured, tools can:
- **Swagger UI** – render the spec as an interactive HTML page where developers can read about each endpoint and even execute test calls.
- **Swagger Editor** – provides a live YAML editor with syntax checking and auto‑completion.
- **Code generators** – create server stubs (Flask, Node.js, Java Spring) and client SDKs (Python, JavaScript, etc.) from the spec. This ensures the implementation matches the contract exactly.

---

## 8. Documenting an API with Swagger Editor (Practical)

### 8.1 The Swagger Editor
The screencast demonstrates the **Swagger Editor** (editor.swagger.io), a web‑based tool for writing and visualising OAS documents. On the left, you write YAML (or JSON); on the right, a live preview of the generated documentation appears.

### 8.2 Creating an API Description for a User Resource
The example models a simple **User** entity with fields `user_id`, `username`, `email`. The YAML document includes:
- **Top‑level metadata:** `openapi: 3.0.3`, title, description, version.
- **Server:** `http://127.0.0.1:5000/` (local development).
- **Paths:**
  - `GET /api/user/{username}` – retrieve a user by username.  
    *Parameters:* `username` in path (string).  
    *Responses:* `200` with JSON body (schema `User`), `404` (not found).
  - `PUT /api/user/{username}` – update user email.  
    *Request body:* JSON with `email`.  
    *Responses:* `200` with updated user, `400` on validation error, `404` if user does not exist.
  - `DELETE /api/user/{username}` – delete a user.  
    *Responses:* `200` on success, `400` if user has dependent records (e.g., articles).
  - `POST /api/user` – create a new user.  
    *Request body:* JSON with `username` and `email`.  
    *Responses:* `201` (created, no body) or `400` with error details.
- **Error format:** A common error structure `{"error_code": "BE1001", "error_message": "Username is required"}`. This is defined as a reusable schema in `components`.

### 8.3 Enhancing Documentation with Diagrams
The lecturer uses **Mermaid.js** to create an entity‑relationship diagram for the database tables and embeds the image into the description field using Markdown syntax (`![ER diagram](https://mermaid.ink/...)`). This demonstrates that OAS descriptions can include rich media to clarify the data model.

### 8.4 Exploring and Testing the API
Once the YAML is written, it can be:
- Downloaded and shared as a file.
- Hosted on a GitHub Gist and viewed via **Swagger UI** (e.g., `petstore.swagger.io/?url=...`).
- Imported into API testing tools like **Insomnia** or **Postman**, which can parse the spec and auto‑generate requests with example values, making it easy to test against a running server.

This workflow embodies the **design‑first** principle: the spec is the single source of truth. A backend developer implements the server to match the spec; a frontend developer generates a client SDK or manually writes fetch calls that match the documented endpoints, inputs, and outputs.

---

## 9. Building REST APIs with Flask-RESTful

### 9.1 Introduction to Flask-RESTful
**Flask‑RESTful** is an extension for Flask that adds support for quickly building REST APIs. It encourages the use of **Resource** classes (one class per resource), automatic method dispatching (`get`, `post`, etc.), request parsing, output formatting (marshalling), and error handling. It reduces boilerplate and enforces consistent patterns.

### 9.2 Setup and Integration
Install with `pip install flask-restful`. In the Flask application factory, create an `Api` object:
```python
from flask_restful import Api
api = Api(app)
```
This `api` object is used to register resource classes with URL endpoints.

### 9.3 Resource Classes and URL Mapping
A resource is a Python class that inherits from `Resource`. Define methods named after HTTP verbs:
```python
from flask_restful import Resource

class UserAPI(Resource):
    def get(self, username):
        # retrieve user
        pass

    def post(self):
        # create user
        pass

    def put(self, username):
        # update user
        pass

    def delete(self, username):
        # delete user
        pass
```
Map the resource to one or more URLs:
```python
api.add_resource(UserAPI, '/api/user', '/api/user/<string:username>')
```
- `GET`, `PUT`, `DELETE` on `/api/user/{username}` will call `get(username)`, `put(username)`, `delete(username)`.
- `POST` on `/api/user` (no username in URL) will call `post()` (no arguments). Flask‑RESTful intelligently routes based on method and URL pattern.

### 9.4 Request Parsing and Validation
Flask‑RESTful provides `reqparse` for parsing and validating incoming request data. However, the screencast shows a custom approach to have full control over the error response format (matching the OAS error schema). A parser is created, arguments added, and then `.parse_args()` returns the parsed values (or `None` if missing). Custom validation is done manually:
```python
parser = reqparse.RequestParser()
parser.add_argument('username')
parser.add_argument('email')
args = parser.parse_args()
username = args.get('username')
email = args.get('email')
if not username:
    raise BusinessValidationError(400, 'BE1001', 'Username is required')
```
This uses a custom exception class `BusinessValidationError` that generates a JSON response with the correct error code and message.

### 9.5 Custom Exception Handling
To return consistent, documented error responses, custom exceptions extending `HTTPException` are defined:
```python
class BusinessValidationError(HTTPException):
    def __init__(self, status_code, error_code, error_message):
        self.response = make_response(
            json.dumps({'error_code': error_code, 'error_message': error_message}),
            status_code
        )
```
When raised, Flask‑RESTful catches these exceptions and sends the JSON response with the specified status code and body. This ensures that all validation errors (missing fields, invalid email format, duplicate users) follow the exact contract defined in the OpenAPI spec.

### 9.6 Output Marshalling with `marshal_with`
To format the response JSON in a consistent way — and to exclude internal fields or format values — Flask‑RESTful provides `marshal_with` decorator and `fields`. A dictionary defines the output structure:
```python
output_fields = {
    'user_id':   fields.Integer,
    'username':  fields.String,
    'email':     fields.String,
}

@marshal_with(output_fields)
def get(self, username):
    user = db.session.query(User).filter_by(username=username).first()
    if not user:
        raise NotFoundError(404)
    return user
```
- The decorator automatically converts the returned object (or dictionary) into a JSON response that contains only the specified fields, in the defined types.
- If `user` is a SQLAlchemy model instance, its attributes are extracted and marshalled.
- This abstraction allows changing the database model without changing the API response format, as long as the marshalling fields map correctly.

### 9.7 Handling HTTP Status Codes
The resource method can return a tuple `(response_body, status_code)`, or raise exceptions that set the status code. For example, on successful creation (`POST`):
```python
def post(self):
    # ... validation ...
    new_user = User(username=username, email=email)
    db.session.add(new_user)
    db.session.commit()
    return '', 201   # Empty body, 201 Created
```
For `DELETE`:
```python
def delete(self, username):
    user = db.session.query(User).filter_by(username=username).first()
    if not user:
        raise NotFoundError(404)
    db.session.delete(user)
    db.session.commit()
    return '', 200
```
The status codes must match the API documentation exactly, ensuring that clients can rely on them for programmatic decisions.

### 9.8 Testing the API with Insomnia
**Insomnia** (or Postman) is a GUI client for testing REST APIs. It can import an OpenAPI specification and auto‑generate requests with pre‑filled URLs, headers, and example bodies. The screencast demonstrates:
- Sending a `GET` request to `/api/user/thejeshgn` and verifying the JSON response matches the marshalled format.
- Sending a `POST` with missing fields and verifying that `400` and the correct error code/message are returned.
- Sending a valid `POST` and verifying a `201` and that the user now appears in a subsequent `GET`.
- Testing `DELETE` and `PUT` similarly.

This testing closes the loop: from **design (OpenAPI spec)** → **implementation (Flask‑RESTful)** → **validation (Insomnia against the spec)**. It guarantees that the API behaves exactly as documented and that client developers can start building their side with confidence.

### 9.9 Integration with SQLAlchemy Models
The API handlers use SQLAlchemy models to query and modify the database, just like in previous weeks. For example:
```python
user = db.session.query(User).filter(User.username == username).first()
```
or using the relationship to enforce constraints. All database operations are wrapped in the session and properly handled for errors (with appropriate status codes). The API layer is the controller: it orchestrates between the HTTP request/response and the model layer, never bypassing the model’s logic.

---

## Summary of Week 6

- **REST** is an architectural style defined by constraints (client‑server, statelessness, cacheability, layered system, uniform interface, code‑on‑demand). It is not a protocol but a set of principles for designing networked applications that scale and evolve.
- **HTTP** provides a natural implementation: URIs identify resources; methods (`GET`, `POST`, `PUT`, `DELETE`) express operations; status codes convey outcomes; idempotency concepts enable robust retry logic.
- **Real‑world APIs** (Wikipedia, CoWin) demonstrate how RESTful design separates data from presentation, provides structured JSON responses, and uses authentication and versioning.
- **OpenAPI Specification (OAS)** is a machine‑readable standard for describing REST APIs. It enables interactive documentation, code generation, and serves as a contract‑first design tool. YAML is the most common format.
- **Swagger tools** (Editor, UI) allow developers to write, visualise, and share API specifications, embedding rich descriptions and diagrams.
- **Flask‑RESTful** extends Flask with a resource‑oriented approach to building REST APIs, including automatic method dispatching, request parsing, output marshalling, and custom exception handling.
- The **practical workflow** demonstrated — write the OpenAPI spec, implement the Flask‑RESTful resources following that spec, and test with an API client — embodies the industry best practice of design‑first, contract‑driven development. This ensures that the API is consistent, well‑documented, and maintainable.

# Modern Application Development – I: Comprehensive Lecture Notes  
**Professor Thejesh G N, IIT Madras**  
**Week 6 (Continuation): Implementing DELETE and PUT in Flask-RESTful**

---

## Table of Contents
1. [Overview: Completing the User CRUD API](#1-overview-completing-the-user-crud-api)
2. [Implementing the DELETE Operation](#2-implementing-the-delete-operation)
3. [Implementing the PUT (Update) Operation](#3-implementing-the-put-update-operation)
4. [Testing the Complete API with Insomnia](#4-testing-the-complete-api-with-insomnia)
5. [Design Decisions and Best Practices](#5-design-decisions-and-best-practices)

---

## 1. Overview: Completing the User CRUD API

In the previous screencast, we built the `GET` and `POST` endpoints for a `User` resource using Flask‑RESTful, following the OpenAPI specification. The remaining CRUD operations are `DELETE` (remove a user) and `PUT` (update a user’s email). This session focuses on implementing these two endpoints with proper validation, error handling, and response formatting, completing a fully functional RESTful API for the `User` entity.

By the end of this part, the API will support:

| Method | URL                    | Action                         |
|--------|------------------------|--------------------------------|
| GET    | /api/user/{username}   | Retrieve a user                |
| POST   | /api/user              | Create a new user              |
| PUT    | /api/user/{username}   | Update an existing user’s email|
| DELETE | /api/user/{username}   | Delete a user (if no articles) |

All responses must adhere to the contract defined in the OpenAPI documentation, including status codes, JSON schemas for success and error, and the standard error format `{"error_code": "…", "error_message": "…"}`.

---

## 2. Implementing the DELETE Operation

### 2.1 API Contract for DELETE
According to the OpenAPI specification designed earlier, the `DELETE /api/user/{username}` endpoint must:
- Return `200 OK` with an empty body if the user is successfully deleted.
- Return `404 Not Found` if the username does not exist.
- Return a business error (e.g., `400 Bad Request`) with an appropriate error code if the user cannot be deleted because of existing dependencies (like articles authored by the user). In our documentation, this error might be defined as `BE1005` with a message indicating that the user has associated articles.

### 2.2 Checking for User Existence
The first step in the `delete` method is to verify that the user actually exists. We reuse the same SQLAlchemy query as in the `GET` method:
```python
def delete(self, username):
    user = db.session.query(User).filter(User.username == username).first()
    if not user:
        raise NotFoundError(status_code=404)
```
If `user` is `None`, we raise the custom `NotFoundError` exception, which returns a `404` response with an empty body. This is consistent with REST principles: the client should be able to distinguish between “the resource does not exist” and “the resource existed but is now gone”.

### 2.3 Checking for Dependencies (Articles)
Before deleting a user, we must check if the user has any associated articles. The `article_authors` table links users to articles, enforcing referential integrity. If we attempt to delete a user who is referenced in `article_authors`, the database would either raise a foreign‑key constraint violation (if `ON DELETE RESTRICT` or similar is set) or cascade the delete (if `ON DELETE CASCADE`). The API’s requirements may dictate whether we allow cascade deletes. In our design, we choose **not to cascade**; instead, we forbid the deletion and inform the client.

We can check for associated articles with a simple query on the `Article` model, leveraging the relationship:
```python
articles = Article.query.filter(Article.authors.any(User.username == username)).first()
if articles:
    raise BusinessValidationError(
        status_code=400,
        error_code='BE1005',
        error_message='Cannot delete user; there are articles written by this user.'
    )
```
- `Article.authors.any(User.username == username)` uses SQLAlchemy’s `any()` to check if the `authors` relationship contains any user with the given username.
- If at least one article exists, we raise a `BusinessValidationError` with a `400` status and a descriptive message. The error code `BE1005` is an extension to the original documentation; in a real project, the OpenAPI spec would define all possible error codes upfront.

**Why check manually instead of catching a database exception?**  
It is often better to perform application‑level validation before hitting the database, because:
- The error message can be more user‑friendly and precisely tailored.
- The check avoids an unnecessary database round‑trip and exception handling overhead.
- It clearly separates business logic from persistence‑layer errors.

However, if multiple concurrent requests could create a race condition (e.g., an article being added between the check and the delete), you might still need to handle the database exception as a fallback.

### 2.4 Performing the Deletion
If no articles are associated, we proceed to delete the user:
```python
db.session.delete(user)
db.session.commit()
return '', 200
```
- `db.session.delete(user)` marks the object for deletion.
- `db.session.commit()` flushes the deletion to the database and makes it permanent.
- The method returns an empty string with HTTP status `200`. Flask‑RESTful automatically creates a response with no body. An alternative status code is `204 No Content`, but the documentation here specifies `200`, so we follow that.

After successful deletion, a subsequent `GET` on the same username should return `404`.

### 2.5 Error Handling in Action
Testing with Insomnia:
- **Existing user with articles** (e.g., `thejeshgn`): returns `400` with `{"error_code": "BE1005", "error_message": "Cannot delete user…"}`.
- **Existing user with no articles** (e.g., `raj5`): returns `200` and the user is removed from the database.
- **Non‑existent user**: returns `404 Not Found`.

This pattern of checking existence → checking dependencies → performing the action keeps the controller clean and the responses predictable.

---

## 3. Implementing the PUT (Update) Operation

### 3.1 API Contract for PUT
The `PUT /api/user/{username}` endpoint updates an existing user’s email address. According to the spec:
- **Request body:** JSON object with a single field `email` (required).
- **Success response:** `200 OK` with the full updated user object (user_id, username, email).
- **Error responses:**
  - `400` if `email` is missing, invalid format, or duplicates an existing email.
  - `404` if the user identified by `username` does not exist.

### 3.2 Parsing the Update Request
We define a new `RequestParser` specific to the update operation, because the `POST` parser also includes `username` (which is not sent in the body for updates – the username comes from the URL). For `PUT`, we only expect `email`:
```python
update_user_parser = reqparse.RequestParser()
update_user_parser.add_argument('email')
```
In the `put` method:
```python
def put(self, username):
    args = update_user_parser.parse_args()
    email = args.get('email')
```

### 3.3 Input Validation
**Missing email:**  
If `email` is `None` or empty:
```python
if not email:
    raise BusinessValidationError(400, 'BE1002', 'Email is required')
```

**Invalid email format:**  
We keep the simple check for an `@` symbol (a more robust validation would use a regular expression or a library):
```python
if '@' not in email:
    raise BusinessValidationError(400, 'BE1003', 'Invalid email')
```

### 3.4 Duplicate Email Check
An update must not set an email that already belongs to another user. We query for any user whose email matches the provided email, excluding the current user (because keeping the same email is harmless):
```python
existing_user = db.session.query(User).filter(
    User.email == email,
    User.username != username
).first()
if existing_user:
    raise BusinessValidationError(400, 'BE1006', 'Duplicate email')
```
This ensures uniqueness across all users except the one being updated.

### 3.5 User Existence Check
We then retrieve the user by `username`. If not found, raise `NotFoundError` (404):
```python
user = User.query.filter_by(username=username).first()
if not user:
    raise NotFoundError(404)
```

### 3.6 Performing the Update
Once validations pass, we update the user’s email and commit:
```python
user.email = email
db.session.add(user)   # user is already tracked; add is harmless but often omitted
db.session.commit()
```
Note: `db.session.add(user)` is not strictly necessary because `user` is a persistent object obtained from a query – it is already in the session. However, it is safe and can make the intent explicit.

### 3.7 Returning the Updated Resource
The spec requires the full updated object. We reuse the `marshal_with` decorator from the `GET` endpoint:
```python
@marshal_with(output_fields)
def put(self, username):
    ...
    return user
```
- The `output_fields` dictionary defines `user_id`, `username`, `email`.
- `marshal_with` automatically serialises the returned `user` object into a JSON response with status `200`.

### 3.8 Error Handling in Action (PUT)
Testing with Insomnia:
- **No email provided:** `400` – `{"error_code": "BE1002", "error_message": "Email is required"}`
- **Invalid email (missing @):** `400` – `BE1003`
- **Duplicate email:** `400` – `BE1006`
- **Non‑existent user:** `404`
- **Valid update:** `200` with the full user JSON, and the database is updated.

The sequence of checks (validation → duplicate check → existence → update) ensures that the most fundamental errors (missing fields) are caught first, avoiding unnecessary database queries.

---

## 4. Testing the Complete API with Insomnia

With all four endpoints implemented, we can now test the entire lifecycle:
1. **Create a user** (`POST /api/user`) – returns `201`, user appears in DB.
2. **Retrieve the user** (`GET /api/user/{username}`) – returns `200` with the created data.
3. **Update the user’s email** (`PUT /api/user/{username}`) – returns `200` with updated data; database reflects change.
4. **Delete the user** (`DELETE /api/user/{username}`) – returns `200`, user removed.
5. **Verify deletion** (`GET` again) – returns `404`.

Insomnia (or any API client) allows us to import the OpenAPI spec (if we had the full spec) or manually create requests. The screencast demonstrates that the implemented API conforms to the documented contract. This contract‑first approach reduces ambiguity and ensures that frontend and backend developers can work independently once the spec is agreed upon.

---

## 5. Design Decisions and Best Practices

### 5.1 Manual Dependency Checks vs. Database Cascades
The lecture chooses to manually check for articles before deleting a user, forbidding deletion if dependencies exist. Alternatives include:
- **ON DELETE CASCADE**: If the foreign key constraint in `article_authors` is set to cascade, deleting a user would automatically delete all associated `article_authors` rows (and possibly articles if cascading further). This can be dangerous if not intended, but it simplifies client logic. The choice depends on the business requirements.
- **Soft delete**: Instead of physically deleting rows, a `deleted` flag is set. This preserves data for auditing and simplifies dependency management (articles can remain with a reference to a “deleted” user). The API would then return `404` for normal queries, but the data persists.

### 5.2 Separation of Concerns in Validation
Validation is kept in the controller (API layer) rather than the model, which is typical for RESTful services. The model layer could also enforce basic constraints (e.g., `nullable=False`, `unique=True`), and the API layer adds business‑specific rules (error codes, message formats). In a larger application, validation logic might be extracted into a separate service layer to avoid duplication between create and update operations.

### 5.3 Consistent Error Response Format
All errors, whether `404` (not found) or `400` (validation), return a JSON body with `error_code` and `error_message`. This uniformity makes client‑side error handling straightforward. The custom exception classes (`NotFoundError`, `BusinessValidationError`) centralise the formatting, ensuring that adding new error cases does not introduce inconsistencies.

### 5.4 Marshalling for Consistent Output
Using `marshal_with` ensures that the API always returns exactly the fields declared in `output_fields`. Even if the internal model gains new columns (e.g., `created_date`), the API will not leak them unless the output fields are updated. This is a powerful encapsulation mechanism.

### 5.5 Adherence to the OpenAPI Contract
Every decision — status codes, response bodies, error codes, URL patterns — is dictated by the OpenAPI specification written before coding. This demonstrates the **design‑first** approach. The specification is the single source of truth; any deviation in the implementation would be a bug.

---

## Summary

In this final part of the Flask‑RESTful implementation, we completed the CRUD interface for the `User` resource by implementing `DELETE` and `PUT`. The key concepts covered include:

- **Dependency checking** before deletion to maintain referential integrity and provide clear client feedback.
- **Update‑specific validation** (missing fields, invalid format, uniqueness constraints) and the order in which checks are performed.
- **Reuse of error handling** via custom exceptions, and output formatting via `marshal_with`.
- **Full‑lifecycle testing** using an API client to verify that the implementation matches the specification exactly.

This completes the practical implementation of a RESTful API using Flask, SQLAlchemy, Flask‑RESTful, and the OpenAPI Specification, tying together all the theoretical principles of MVC, REST constraints, and API design into a working, documented, and testable web service.